<a href="https://colab.research.google.com/github/801-Hillside-Terrace/SMART-2026/blob/main/week7/week7_NeuralNetworkTraining.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
#imports
import math
import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.optim as optim
import csv
from sklearn.datasets import load_iris
from torch.utils.data import TensorDataset, DataLoader


# set seed
torch.manual_seed(801)

# load data
iris = load_iris()
X = iris['data']
y = iris['target']

# convert to tensors
X = torch.tensor(X, dtype=torch.float32)
y = torch.tensor(y, dtype=torch.float32)


# split and standardize
def train_test_split(X, y, test_pct=0.2, seed=801):
    torch.manual_seed(seed)
    n = X.shape[0]
    perm = torch.randperm(n)

    split = int(n * (1 - test_pct))
    train_idx = perm[:split]
    test_idx = perm[split:]

    return X[train_idx], X[test_idx], y[train_idx], y[test_idx]

def standardize_train_test(X_train, X_test):
    mean = X_train.mean(dim=0, keepdim=True)
    std = X_train.std(dim=0, keepdim=True) + 1e-8

    X_train_scaled = (X_train - mean) / std
    X_test_scaled = (X_test - mean) / std

    return X_train_scaled, X_test_scaled

X_train, X_val, y_train, y_val = train_test_split(X, y)
X_train, X_val = standardize_train_test(X_train, X_val)

num_input = X_train.shape[1]
num_output = y_val.unique().numel()

model = nn.Sequential(
    nn.Linear(num_input, 10),
    nn.ReLU(),
    nn.Linear(10, num_output)
)

The first trick is to use tensor datasets and dataloaders.  The TensorDataset keeps everything organized while the dataloaders help us automate the batching and shuffling.  Shuffling changes what is in the batches each epoch (so it isnt the same batches every time).  They should also be faster/more efficient than manual batching.

In [2]:
# TensorDataset just takes (X,y)
train_dataset = TensorDataset(X_train, y_train)
val_dataset = TensorDataset(X_val, y_val)

# in the dataloaders we feed in the TensorDataset and set the batch size and whether or not we shuffle, shuffling is typically only do to the training loader (shuffles what is in each batch at each epoch)
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=32, shuffle=False)

Question: Why do we shuffle the training but not the val/test?

Recall the momentum concept from Week 4.  With momentum, we effectively include an average of the past gradients in our update proportional to the momentum we set.  IE momentum of 0.9 means 10% of the update uses the current gradient while 90% is from the past gradients averaged in a way that decays exponentially.  This helps by speeding up if we are in a good direction and prevents drastically switching directions too often.

A more modern approach is adaptive moment estimation with decoupled weight decay otherwise known as "AdamW".  https://optimization.cbe.cornell.edu/index.php?title=AdamW

This optimizer uses momentum, adaptive gradients, and weight decay in a workable manner (Adam predates AdamW but implements weight decay incorrectly).  The adaptive gradient part works by scaling the gradients down proportional to their average squared size.  IE if a certain weight often gets large gradients, we scale down its updates to stop it from blowing up too fast and if a different weight often gets small gradients, we scale up its updates so that it doesn't stagnate.  The update for a certain weight is then proportional to momentum/typical gradient size.  In practice this is like giving each weight its own learning rate that adapts to how fast it is growing in a way that keeps everything better behaved.  So we get the smoothing and acceleration in good directions from momentum, but we also shrink steps that are in steep directions and enlarge steps that are in flat directions.  

The main downside is that Adam/AdamW may adapt too aggressively, and so well-tuned SGD with momentum may be able to generalize better.  This is probably a question for Greg though I think he told me that one.

In [3]:
# here is an example of setting AdamW as the optimizer with lr = 0.001 and lambda = 0.0001
# the defaults for the momentum and adaptive aspect are 0.9 and 0.999 respectively and tend to work well
optimizer = optim.AdamW(
    model.parameters(),
    lr=1e-3,
    weight_decay=1e-4
)

Next we have batch and layer normalization.  We can add this into nn.Sequential(), it normalizes each feature within the batch (subtracts mean and divides by SD) and then scales them by a coefficient and an intercept which is learned.  It also tracks the running mean and SD and uses these when we switch to evaluation.  Layernorm is similar, but it does this row-wise instead and does not track any running averages or change during evaluation.  In practice this seems to help training, but with batchnorm we have to be careful because things can break when batch sizes are very small and if the running statistics differ greatly between train and val/test.    

In [4]:
model_batch_norm = nn.Sequential(
    nn.Linear(num_input, 10),
    nn.BatchNorm1d(10), # adding batchnorm (typically done before activation)
    nn.ReLU(),
    nn.Linear(10, num_output)
)

Learning rate schedulers are another tool that is used, in general you want a higher learning rate at the start of training and a smaller one later on.  Cosine annealing has the learning rate smoothly decrease and is currently popular.  When using one, depending on the type, you must update it every epoch or every batch (so that it updates the learning rate at the right time).  Cosine annealing is updated per epoch.

In [5]:
scheduler = optim.lr_scheduler.CosineAnnealingLR(
    optimizer, # must give it the optimizer to update the optimizer's learning rate
    T_max=100 # epochs until hitting minimum
)
# we then need scheduler.step() at the end of every epoch loop

In [6]:
'''
# Classification Example

device = torch.device("cuda" if torch.cuda.is_available() else "cpu") # set device to cuda if we are using it

X_train = X_train.float() # typically floats
X_val = X_val.float()

y_train = y_train.long() # y values should be longs for cross entropy loss
y_val = y_val.long()

batch_size = 32 # set fixed batch size

train_dataset = TensorDataset(X_train, y_train) # tensor datasets
val_dataset   = TensorDataset(X_val, y_val)

# data loaders
train_loader = DataLoader(train_dataset, batch_size = batch_size, shuffle = True)
val_loader = DataLoader(val_dataset, batch_size = batch_size, shuffle = False)

# input and output dimension calculations
input_dim = X_train.shape[1] # number of features
output_dim = y_train.unique().numel() # number of unique y values

# model
model = nn.Sequential(
    nn.Linear(input_dim, o1),
    nn.BatchNorm1d(o1),
    nn.ReLU(), # can try GELU too
    nn.Dropout(p1),

    nn.Linear(o1, o2),
    nn.LayerNorm(o2), # in practice probably dont mix these
    nn.ReLU(),
    nn.Dropout(p2),

    nn.Linear(o2, o3),
    nn.ReLU(),

    nn.Linear(o3, output_dim) # output is number of classes (one logit per)
).to(device)

# loss function
criterion = nn.CrossEntropyLoss() # for classification, automatically computes softmax so we don't need that in the model itself etc

# optimizer
optimizer = optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-4)

num_epochs = 50

# learning rate scheduler if we want it
#scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max = num_epochs)

for epoch in range(num_epochs):

    model.train()

    train_loss = 0.0
    train_correct = 0
    train_total = 0

    for xb, yb in train_loader:
        optimizer.zero_grad()

        xb = xb.to(device)
        yb = yb.to(device)

        # forward pass
        preds = model(xb) # raw logits

        loss = criterion(preds, yb) # note that yb must have shape (b,) while preds have shape (b,output_dim)

        # backward pass
        loss.backward()

        optimizer.step()

        # loss and accuracy tracking
        train_loss += loss.item() * xb.shape[0]

        predicted = preds.argmax(dim=1) # predictions are largest value along dimension 1 (column with largest logit)

        train_correct += (predicted == yb).sum().item()

        train_total += yb.shape[0]

    train_loss /= train_total
    train_acc = train_correct / train_total

    # validation
    model.eval()

    val_loss = 0.0
    val_correct = 0
    val_total = 0

    with torch.no_grad():

        for xb, yb in val_loader:

            xb = xb.to(device)
            yb = yb.to(device)

            preds = model(xb)

            loss = criterion(preds, yb)

            val_loss += loss.item() * xb.shape[0]

            predicted = preds.argmax(dim=1)

            val_correct += (predicted == yb).sum().item()

            val_total += yb.shape[0]

    val_loss /= val_total
    val_acc = val_correct / val_total

    # update scheduler if using
    #scheduler.step()

    # print each epoch results
    print(f"Epoch {epoch+1} | Train Loss: {train_loss:.4f} | Train Acc: {train_acc:.4f} | Val Loss: {val_loss:.4f} | Val Acc: {val_acc:.4f}")

'''

'\n# Classification Example\n\ndevice = torch.device("cuda" if torch.cuda.is_available() else "cpu") # set device to cuda if we are using it\n\nX_train = X_train.float() # typically floats\nX_val = X_val.float()\n\ny_train = y_train.long() # y values should be longs for cross entropy loss\ny_val = y_val.long()\n\nbatch_size = 32 # set fixed batch size\n\ntrain_dataset = TensorDataset(X_train, y_train) # tensor datasets\nval_dataset   = TensorDataset(X_val, y_val)\n\n# data loaders\ntrain_loader = DataLoader(train_dataset, batch_size = batch_size, shuffle = True)\nval_loader = DataLoader(val_dataset, batch_size = batch_size, shuffle = False)\n\n# input and output dimension calculations\ninput_dim = X_train.shape[1] # number of features\noutput_dim = y_train.unique().numel() # number of unique y values \n\n# model\nmodel = nn.Sequential(\n    nn.Linear(input_dim, o1),\n    nn.BatchNorm1d(o1),\n    nn.ReLU(), # can try GELU too\n    nn.Dropout(p1),\n\n    nn.Linear(o1, o2),\n    nn.LayerN

In [7]:
'''
# Regression Example

device = torch.device("cuda" if torch.cuda.is_available() else "cpu") # set device to cuda if we are using it

X_train = X_train.float() # typically floats
X_val = X_val.float()

y_train = y_train.float().view(-1,1) # targets should be floats, can also reshape to (n,1)
y_val = y_val.float().view(-1,1)

batch_size = 32 # set fixed batch size

train_dataset = TensorDataset(X_train, y_train) # tensor datasets
val_dataset   = TensorDataset(X_val, y_val)

# data loaders
train_loader = DataLoader(train_dataset, batch_size = batch_size, shuffle = True)
val_loader = DataLoader(val_dataset, batch_size = batch_size, shuffle = False)

# input dimension calculation
input_dim = X_train.shape[1] # number of features

# model
model = nn.Sequential(
    nn.Linear(input_dim, o1),
    nn.BatchNorm1d(o1),
    nn.ReLU(), # can try GELU too
    nn.Dropout(p1),

    nn.Linear(o1, o2),
    nn.LayerNorm(o2), # in practice probably dont mix these
    nn.ReLU(),
    nn.Dropout(p2),

    nn.Linear(o2, o3),
    nn.ReLU(),

    nn.Linear(o3, 1) # final output is 1 for regression (predicting single scalar value)
).to(device)

# loss function
criterion = nn.MSELoss() # for regression

# optimizer
optimizer = optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-4)

num_epochs = 50

# learning rate scheduler if we want it
#scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max = num_epochs)

for epoch in range(num_epochs):

    model.train()

    train_loss = 0.0
    train_total = 0

    for xb, yb in train_loader:
        optimizer.zero_grad()

        xb = xb.to(device)
        yb = yb.to(device)

        # forward pass
        preds = model(xb)

        loss = criterion(preds, yb) # note that preds and yb must have the same shape of both either (b,) or (b,1)

        # backward pass
        loss.backward()

        optimizer.step()

        # loss tracking
        train_loss += loss.item() * xb.shape[0]

        train_total += yb.shape[0]

    train_loss /= train_total

    # validation
    model.eval()

    val_loss = 0.0
    val_total = 0

    with torch.no_grad():

        for xb, yb in val_loader:

            xb = xb.to(device)
            yb = yb.to(device)

            preds = model(xb)

            loss = criterion(preds, yb)

            val_loss += loss.item() * xb.shape[0]

            val_total += yb.shape[0]

    val_loss /= val_total

    # update scheduler if using
    #scheduler.step()

    # print each epoch results
    print(f"Epoch {epoch+1} | Train Loss: {train_loss:.4f} | Val Loss: {val_loss:.4f}")

    '''

'\n# Regression Example\n\ndevice = torch.device("cuda" if torch.cuda.is_available() else "cpu") # set device to cuda if we are using it\n\nX_train = X_train.float() # typically floats\nX_val = X_val.float()\n\ny_train = y_train.float().view(-1,1) # targets should be floats, can also reshape to (n,1)\ny_val = y_val.float().view(-1,1)\n\nbatch_size = 32 # set fixed batch size\n\ntrain_dataset = TensorDataset(X_train, y_train) # tensor datasets\nval_dataset   = TensorDataset(X_val, y_val)\n\n# data loaders\ntrain_loader = DataLoader(train_dataset, batch_size = batch_size, shuffle = True)\nval_loader = DataLoader(val_dataset, batch_size = batch_size, shuffle = False)\n\n# input dimension calculation\ninput_dim = X_train.shape[1] # number of features\n\n# model\nmodel = nn.Sequential(\n    nn.Linear(input_dim, o1),\n    nn.BatchNorm1d(o1),\n    nn.ReLU(), # can try GELU too\n    nn.Dropout(p1),\n\n    nn.Linear(o1, o2),\n    nn.LayerNorm(o2), # in practice probably dont mix these\n    nn.R